In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
train_lines_in = "../data/raw/train_lines.geojson"
train_stations_in = "../data/interim/train_stations.geojson"
bike_stations_file_in = "../data/interim/bike_stations_gbfs_v2.geojson"
tract_file_in = "../data/interim/tracts.geoparquet"

train_rides_in = "../data/interim/train_rides.csv"
bike_rides_in = "../data/interim/bike_rides_v4.geoparquet"
uber_tract_rides_in = "../data/interim/uber_tract_rides.parquet"

train_station_rides_out = "../data/final/train_station_rides.parquet"
bike_rides_out = "../data/final/bike_rides.geoparquet"
uber_tract_rides_out = "../data/final/uber_tract_rides.parquet"

# Pipeline in

In [3]:
train_lines = gpd.read_file(train_lines_in)
train_stations = gpd.read_file(train_stations_in)
tracts = gpd.read_parquet(tract_file_in)
bike_stations = gpd.read_file(bike_stations_file_in)

In [4]:
train_rides = pd.read_csv(train_rides_in)
bike_rides = gpd.read_parquet(bike_rides_in)
uber_tract_rides = pd.read_parquet(uber_tract_rides_in)

# Train Rides + Stations

In [5]:
# We only care about granularity of entrypoints, not line/direction because we dont know
# which line/direction the riders go.
train_stations = train_stations.drop_duplicates('map_id')

In [6]:
# Commented out because we are not disaggregating by line.
# # Recover statistic before exploding
# lines_per_station = train_stations.set_index('map_id')['line'].str.count(',')+1

# # Explode multi-line stations to keep correct cardinality of lines
# train_stations['line'] = train_stations['line'].str.split(',')
# train_stations = train_stations.explode('line')
# train_stations['id'] = train_stations['map_id'].astype(str).str.cat(train_stations['line'], sep="_")

In [7]:
train_rides = train_stations.merge(train_rides, how='right', right_on='station_id', left_on='map_id', validate="1:m")

In [8]:
# Commented out because we are not disaggregating by line.
# # Disaggregate rides evenly
# train_rides['rides'] = train_rides['rides'] / train_rides['map_id'].apply(lines_per_station.get)

# Bike Rides + Stations

In [9]:
# # Double-checking if all bike stations (for the years currently accessed)
# #       already have valid (ie chicago?) geometries
# from shapely import box
# chi_boundary = gpd.read_file(CHI_BOUNDARY_FILE)
# chi_bound_geo = chi_boundary['geometry'].to_crs(LOCAL_CRS).iloc[0]
# chi_box = box(*chi_bound_geo.bounds) # Using bounding box for faster computation
# is_valid = bike_rides['geometry'].to_crs(LOCAL_CRS).within(chi_box)

# # "Invalid" geos are all in Evanston essentially, which is fine for our purposes.
# fig = chi_boundary.boundary.plot()
# bike_rides[~is_valid]['geometry'].drop_duplicates().plot(ax=fig, color='red')

In [10]:
print("Columns to gain by merging: ", set(bike_stations.columns) - set(bike_rides.columns))

Columns to gain by merging:  {'comm_area', 'airport', 'station_id'}


XXX:

For now, since all data vintages currently pulled are denormalized and already
have valid geometries, on which we've already computed our spatial features,
there's actually nothing to merge via bike_stations.

If you return to this, note that within bike_stations the station_id <-> geometry is not 1:1,
even though according to the documentation each station has multiple bike docks.
Some of the points are thousands of feet away per station_id.
Therefore be careful about how you construct the merge.

# Uber Rides + Tracts

XXX:
Not dropping anonymized observations (null tract/comm) because they still add to the non-spatial
time series count.

In [11]:
assert not tracts.duplicated(subset='geoid10').any()
assert not uber_tract_rides.duplicated(subset=['tract','date']).any()

In [12]:
uber_tract_rides = tracts.merge(uber_tract_rides, how='right', left_on='geoid10', right_on='tract')

# Pipeline out

In [13]:
uber_tract_rides.to_parquet(uber_tract_rides_out)
bike_rides.to_parquet(bike_rides_out)
train_rides.to_parquet(train_station_rides_out)